# Subir los datos a Databricks
Se ejecuta **una sola vez en su computadora, antes de la simulación**. Crea el esquema `sismos` con sus Volumes y sube los datos que usa el pipeline desde la carpeta local `raw/` (la llena `descarga.ipynb`):
- **Maestros** → `landing/maestros/`: los lee directamente `01_bronze`.
- **Acelerogramas de CISMID** de los 5 sismos → `fuente/cismid/<evento>/`: quedan guardados en Databricks y `Simulacion_Llegada_Datos` los copia a `landing/sismos/` uno por uno, como si fueran llegando.

STEAD pesa 15 GB: se sube una muestra reproducible de 10 000 trazas con onda P y onda S (la misma cantidad que usa el detector). Si se vuelve a ejecutar, solo sube los archivos que faltan o que cambiaron de tamaño.

## 1. Conexión a Databricks

In [ ]:
# Librerías y conexión al workspace de Databricks.
from pathlib import Path

import h5py
import pandas as pd
from databricks.sdk import WorkspaceClient
from databricks.sdk.errors import NotFound
from databricks.sdk.service.catalog import VolumeType

PERFIL = "22200144"
w = WorkspaceClient(profile=PERFIL)
print("Conectado como", w.current_user.me().user_name)

## 2. Configuración

In [ ]:
# Rutas locales, Volumes y maestros que se suben.
CATALOGO = "grupo06proyectofinal"
ESQUEMA = "sismos"
VOLUMENES = ["landing", "fuente", "modelos", "checkpoints"]
CARPETA_LOCAL = Path("raw")
RUTA_VOLUMENES = f"/Volumes/{CATALOGO}/{ESQUEMA}"
MAESTROS = {
    "vs30/global_vs30.grd": "vs30/global_vs30.grd",
    "limites_inei": "limites_inei",
    "limites_osm": "limites_osm",
    "microzonificacion": "microzonificacion",
    "microzonificacion_cismid": "microzonificacion_cismid",
    "open_buildings": "open_buildings",
    "open_buildings_25d": "open_buildings_25d",
    "norma_e030": "norma_e030",
    "censo2017": "censo2017",
    "osm_niveles/lima_building_levels.json": "osm_niveles/lima_building_levels.json",
    "tainan_2016": "tainan_2016",
    "portoviejo_2016/campo/parameters_priority_index_metadata.csv": "portoviejo_2016/campo/parameters_priority_index_metadata.csv",
    "manta_2016/ondas/datos": "manta_2016/ondas/datos",
    "bssa14": "bssa14",
    "fragilidad_gem": "fragilidad_gem",
    "stead_muestra": "stead",
}
EVENTOS = sorted(p.name for p in (CARPETA_LOCAL / "cismid").iterdir() if p.is_dir())
print("Sismos locales:", EVENTOS)

## 3. Funciones

In [ ]:
# Crea el esquema y los Volumes si no existen.
def preparar_entorno():
    if ESQUEMA not in {s.name for s in w.schemas.list(catalog_name=CATALOGO)}:
        w.schemas.create(name=ESQUEMA, catalog_name=CATALOGO)
    existentes = {v.name for v in w.volumes.list(catalog_name=CATALOGO, schema_name=ESQUEMA)}
    for volumen in VOLUMENES:
        if volumen not in existentes:
            w.volumes.create(catalog_name=CATALOGO, schema_name=ESQUEMA, name=volumen, volume_type=VolumeType.MANAGED)
    print(f"Esquema {CATALOGO}.{ESQUEMA} con Volumes:", sorted(v.name for v in w.volumes.list(catalog_name=CATALOGO, schema_name=ESQUEMA)))


# Extrae la muestra de STEAD y la guarda como CSV y HDF5.
def preparar_muestra_stead(n=10_000, semilla=1):
    origen, destino = CARPETA_LOCAL / "stead" / "chunk2", CARPETA_LOCAL / "stead_muestra"
    if (destino / "chunk2_muestra.hdf5").exists():
        print("La muestra de STEAD ya existe:", sorted(p.name for p in destino.iterdir()))
        return
    meta = pd.read_csv(origen / "chunk2.csv", low_memory=False)
    aptas = meta[(meta.trace_category == "earthquake_local") & meta.p_arrival_sample.between(600, 5700)
                 & meta.s_arrival_sample.notna() & meta.source_magnitude.notna()]
    muestra = aptas.sample(n=min(n, len(aptas)), random_state=semilla)
    destino.mkdir(exist_ok=True)
    muestra.to_csv(destino / "chunk2_muestra.csv", index=False)
    with h5py.File(origen / "chunk2.hdf5", "r") as h, h5py.File(destino / "chunk2_muestra.hdf5", "w") as salida:
        for nombre in muestra.trace_name:
            original = h["data"][nombre]
            copia = salida.create_dataset(f"data/{nombre}", data=original[()], compression="gzip")
            for clave, valor in original.attrs.items():
                copia.attrs[clave] = valor
    print(f"Muestra de STEAD: {len(muestra):,} de {len(aptas):,} trazas aptas")


# Sube un archivo al Volume si no está o cambió.
def subir_archivo(origen, destino):
    try:
        if w.files.get_metadata(destino).content_length == origen.stat().st_size:
            return False
    except NotFound:
        pass
    with open(origen, "rb") as f:
        w.files.upload(destino, f, overwrite=True)
    return True


# Sube una carpeta al Volume.
def subir_carpeta(local, remoto):
    ruta = CARPETA_LOCAL / local
    archivos = [ruta] if ruta.is_file() else sorted(p for p in ruta.rglob("*") if p.is_file())
    subidos = 0
    for archivo in archivos:
        relativo = "" if ruta.is_file() else "/" + archivo.relative_to(ruta).as_posix()
        subidos += subir_archivo(archivo, f"{RUTA_VOLUMENES}/{remoto}{relativo}")
    print(f"  {remoto}: {len(archivos)} archivos ({subidos} subidos, {len(archivos) - subidos} ya estaban)")


# Sube los maestros a landing/maestros.
def subir_maestros():
    for local, remoto in MAESTROS.items():
        subir_carpeta(local, f"landing/maestros/{remoto}")


# Sube los acelerogramas a fuente/cismid.
def subir_sismos():
    for evento in EVENTOS:
        subir_carpeta(f"cismid/{evento}", f"fuente/cismid/{evento}")


# Lista el contenido de una carpeta del Volume.
def ver_volume(subcarpeta):
    filas = [{"ruta": e.path.replace(RUTA_VOLUMENES + "/", ""), "carpeta": e.is_directory, "mb": round((e.file_size or 0) / 1e6, 1)}
             for e in w.files.list_directory_contents(f"{RUTA_VOLUMENES}/{subcarpeta}")]
    display(pd.DataFrame(filas))

## 4. Esquema y Volumes

In [ ]:
# Crea el esquema y los Volumes.
preparar_entorno()

## 5. Muestra de STEAD

In [ ]:
# Toma la muestra de 10 000 trazas de STEAD.
preparar_muestra_stead()

## 6. Maestros → landing/maestros

In [ ]:
# Sube los maestros.
subir_maestros()

## 7. Acelerogramas de los 5 sismos → fuente/cismid

In [ ]:
# Sube los acelerogramas de los sismos.
subir_sismos()

## 8. Verificación

In [ ]:
# Lista lo que quedó en los Volumes.
ver_volume("landing/maestros")
ver_volume("fuente/cismid")